# 01 Data Profile

**Question:** What's in the data, and is it clean?

Task 2, step 1: profile the data. This covers **row counts per log and per day, missing values, duplicates, value ranges and the number of unique values per column**, for all four logs (bids, impressions, clicks, conversions) over the 7 days, 6–12 June 2013. **Output:** one data profile table, saved as `data/processed/data_profile.csv`.

**Before you run this:** the loading cells must already have run. They build the `raw_imp`, `raw_clk` and `raw_conv` tables and the `typed_*` views in `data/processed/ipinyou.duckdb`, and the bid Parquet files in `data/interim/bid_parquet`. This notebook only reads: the database is opened **read-only**, so it needs no free disk space. The one exception is the small profile CSV, and it skips that if the disk is full.

**How missing values are counted:** a value is *missing* if it is a real NULL, an empty field, or the literal text `null`. iPinYou writes `null` as text, which is why the typed views turn it into NULL.

### Setup

In [3]:
# CELL 1: setup (read-only connection)
from pathlib import Path
import os, re, shutil, signal, time
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)

KILL_STALE = False          # True only if a stray notebook holds the lock (its unsaved work is lost)
EXACT_DISTINCT_FOR_BIDS = False   # bids: approximate distinct counts (HyperLogLog, ~2% error) to save memory

PROJECT_ROOT = Path.cwd().parent
# Where the data comes from (the views raw_* and typed_* are the same either way):
#   1. data/processed/ipinyou.duckdb, the database the loading notebook built, if it is still there
#   2. otherwise the pipeline's hourly Parquet files in data/interim, read in memory (pipeline_views.py)
DB_PATH = PROJECT_ROOT / "data" / "processed" / "ipinyou.duckdb"
BID_DIR = PROJECT_ROOT / "data" / "interim" / "bid_parquet"
LOGS = ["bid", "imp", "clk", "conv"]
INT_COLS = ["log_type", "region", "city", "ad_exchange", "ad_slot_width", "ad_slot_height",
            "ad_slot_visibility", "ad_slot_format", "ad_slot_floor_price",
            "bidding_price", "paying_price", "advertiser_id"]

print(f"Free disk space: {shutil.disk_usage(PROJECT_ROOT).free / 1e9:.1f} GB")
if "con" in globals():
    try:
        con.close()
    except Exception:
        pass

def open_db(path, kill_stale=False, retries=5):
    for attempt in range(retries):
        try:
            return duckdb.connect(str(path), read_only=True)
        except duckdb.IOException as e:
            m = re.search(r"PID (\d+)", str(e))
            if "lock" not in str(e).lower() or m is None:
                raise
            print(f"The database is locked by another Python process (PID {m.group(1)}).")
            if not kill_stale:
                print("Shut down the other notebook (Jupyter > Running tab > Shut Down), "
                      "or set KILL_STALE = True and run this cell again.")
                raise
            os.kill(int(m.group(1)), signal.SIGTERM)
            time.sleep(2)
    raise RuntimeError("Could not get the database lock")

if DB_PATH.exists():
    con = open_db(DB_PATH, kill_stale=KILL_STALE)
    DATA_SOURCE = f"{DB_PATH.name} (the loading notebook's database)"
else:
    from pipeline_views import create_views
    con = duckdb.connect()   # in memory: reads the Parquet files, writes nothing to disk
    DATA_SOURCE = create_views(con, PROJECT_ROOT)
con.execute("SET preserve_insertion_order=false")
try:
    con.execute("SET enable_progress_bar=false")
except Exception:
    pass

def q(sql):
    return con.execute(sql).df()

# Bids: use the views from the loading notebook, or rebuild them in memory from the Parquet files
views = set(q("SELECT view_name FROM duckdb_views() WHERE NOT internal")["view_name"])
assert {"typed_imp", "typed_clk", "typed_conv"} <= views, "Run the loading cells first (typed_* views missing)"
if not {"raw_bid", "typed_bid"} <= views:
    files = [str(p) for p in sorted(BID_DIR.glob("bid_*.parquet"))]
    files += [str(p) for p in sorted(BID_DIR.glob("bid_*/part_*.parquet")) if not p.parent.name.endswith(".partial")]
    assert files, f"No bid Parquet files in {BID_DIR}"
    con.execute("CREATE OR REPLACE TEMP VIEW raw_bid AS SELECT * FROM read_parquet(["
                + ", ".join(f"'{x}'" for x in files) + "])")
    cols = [c for c in q("DESCRIBE raw_bid")["column_name"] if c != "filename"]
    def typed_expr(c):
        t = f"""CASE WHEN lower("{c}") IN ('', 'null') THEN NULL ELSE "{c}" END"""
        if c == "timestamp":
            return f"TRY_STRPTIME({t}, '%Y%m%d%H%M%S%g')"
        return f"TRY_CAST({t} AS BIGINT)" if c in INT_COLS else t
    con.execute("CREATE OR REPLACE TEMP VIEW typed_bid AS SELECT "
                + ", ".join(f'{typed_expr(c)} AS "{c}"' for c in cols)
                + ", regexp_extract(filename, '([0-9]+)\\.txt$', 1) AS log_date FROM raw_bid")
    print("raw_bid / typed_bid rebuilt in memory from", len(files), "Parquet files")
COLS = {t: [c for c in q(f"DESCRIBE raw_{t}")["column_name"] if c != "filename"] for t in LOGS}
print("Data:", DATA_SOURCE)
print("Columns per log:", {t: len(c) for t, c in COLS.items()})

Free disk space: 3.5 GB
Data: ipinyou.duckdb (the loading notebook's database)
Columns per log: {'bid': 21, 'imp': 24, 'clk': 24, 'conv': 24}


### 1. Row counts per log and per day

A missing or unusually small day means a file didn't load. Expect 7 days in every log.

In [4]:
# CELL 2: rows per log per day, and per campaign
per_day = q(" UNION ALL ".join(
    f"SELECT '{t}' AS log, log_date AS day, count(*) AS n FROM typed_{t} GROUP BY ALL" for t in LOGS))
rows_per_day = per_day.pivot_table(index="day", columns="log", values="n", fill_value=0)[LOGS].astype("int64")
rows_per_day.loc["TOTAL"] = rows_per_day.sum()
display(rows_per_day.style.format("{:,}"))

per_adv = q(" UNION ALL ".join(
    f"SELECT '{t}' AS log, advertiser_id, count(*) AS n FROM typed_{t} GROUP BY ALL" for t in LOGS))
rows_per_adv = per_adv.pivot_table(index="advertiser_id", columns="log", values="n", fill_value=0)[LOGS].astype("int64")
rows_per_adv.loc["TOTAL"] = rows_per_adv.sum()
display(rows_per_adv.style.format("{:,}"))

days_found = sorted(per_day.day.unique())
print("Days found:", days_found, "| all 7 present in every log:",
      bool((rows_per_day.drop(index="TOTAL") > 0).all().all()) and len(days_found) == 7)

log,bid,imp,clk,conv
day,,,,
20130606,"9,586,949","1,821,350","1,289",43
20130607,"11,132,555","1,805,953","1,158",50
20130608,"5,226,937","1,634,830","1,302",52
20130609,"11,880,893","1,651,524","1,250",32
20130610,"5,610,729","1,920,370","1,779",150
20130611,"5,094,845","1,745,722","1,593",61
20130612,"4,756,422","1,657,338","1,607",106
TOTAL,"53,289,330","12,237,087","9,978",494


log,bid,imp,clk,conv
advertiser_id,,,,
1458,"14,701,496","3,083,056","2,610",1
3358,"3,751,016","1,742,104","1,666",466
3386,"14,091,931","2,847,802","2,431",0
3427,"14,032,619","2,593,765","2,179",0
3476,"6,712,268","1,970,360","1,092",27
TOTAL,"53,289,330","12,237,087","9,978",494


Days found: ['20130606', '20130607', '20130608', '20130609', '20130610', '20130611', '20130612'] | all 7 present in every log: True


### 2. Column profile: missing values, unique values, value ranges

For every column of every log:
* **missing**, split into true NULL, empty field, the literal text `null`, and *cast failed* (text that didn't convert to a number or timestamp; this should be 0)
* **distinct**: the number of unique values, its share of non-missing rows (*cardinality*), and the most common value
* **range**: min / max (plus mean and median for numbers)

The note column flags what to drop from reporting: **all missing**, **constant** (one value), and **ID-like** (more than half the values unique: these are join keys, not features).

In [5]:
# CELL 3: the column profile (one scan per log for counts, then exact distinct counts column by column)
def profile_log(t):
    raw, typed, cols = f"raw_{t}", f"typed_{t}", COLS[t]
    n = q(f"SELECT count(*) AS n FROM {raw}")["n"][0]
    # missing split, from the raw text
    parts = []
    for c in cols:
        parts += [f'count(*) FILTER (WHERE "{c}" IS NULL) AS "{c}__null"',
                  f"""count(*) FILTER (WHERE "{c}" = '') AS "{c}__empty" """,
                  f"""count(*) FILTER (WHERE lower("{c}") = 'null') AS "{c}__nulltext" """]
    miss = q(f"SELECT {', '.join(parts)} FROM {raw}").iloc[0]
    dtypes = dict(q(f"DESCRIBE {typed}")[["column_name", "column_type"]].values)
    rows = []
    for c in cols:
        numeric = dtypes[c] in ("BIGINT", "INTEGER", "DOUBLE")
        is_ts = dtypes[c].startswith("TIMESTAMP")
        exact = t != "bid" or EXACT_DISTINCT_FOR_BIDS
        distinct = f'count(DISTINCT "{c}")' if exact else f'approx_count_distinct("{c}")'
        rng = (f', min("{c}")::VARCHAR, max("{c}")::VARCHAR' if (numeric or is_ts) else ", NULL, NULL")
        stats = (f', avg("{c}"), approx_quantile("{c}", 0.5)' if numeric else ", NULL, NULL")
        nn, nd, mn, mx, mean, med = con.execute(
            f'SELECT count("{c}"), {distinct}{rng}{stats} FROM {typed}').fetchone()
        rows.append({"log": t, "column": c, "dtype": dtypes[c], "rows": n,
                     "null": int(miss[f"{c}__null"]), "empty": int(miss[f"{c}__empty"]),
                     "null_text": int(miss[f"{c}__nulltext"]), "missing": n - nn,
                     "missing_pct": round(100 * (n - nn) / n, 2) if n else np.nan,
                     "distinct": int(nd or 0), "distinct_exact": exact,
                     "min": mn, "max": mx,
                     "mean": None if mean is None else round(mean, 2),
                     "median": None if med is None else round(float(med), 2)})
    p = pd.DataFrame(rows)
    p["cardinality"] = (p.distinct / (p.rows - p.missing).replace(0, np.nan)).clip(upper=1).round(4)
    # typed NULL although the raw text had a value = the type conversion failed (should be 0)
    p["cast_failed"] = p.missing - p.null - p.empty - p.null_text
    # most common value (skipped for ID-like columns: it would be meaningless and memory-hungry)
    low = p[(p.cardinality <= 0.5) & (p.distinct > 0)]["column"].tolist()
    if low:
        tops = q(f"SELECT {', '.join(f'mode({chr(34)}{c}{chr(34)})::VARCHAR AS {chr(34)}{c}{chr(34)}' for c in low)} FROM {typed}").iloc[0]
        p["top_value"] = p["column"].map(lambda c: tops.get(c) if c in low else None)
        share = q(" UNION ALL ".join(
            f"""SELECT '{c}' AS column, count(*) FILTER (WHERE "{c}"::VARCHAR = '{str(tops[c]).replace("'", "''")}') / count(*) AS s FROM {typed}"""
            for c in low if tops[c] is not None))
        p["top_share_pct"] = p["column"].map(dict(zip(share["column"], (100 * share.s).round(2))))
    else:
        p["top_value"], p["top_share_pct"] = None, np.nan

    def note(r):
        if r.missing == r.rows:
            return "all missing: drop"
        if r.distinct <= 1:
            return "constant: drop"
        if r.column not in ("timestamp",) and r.cardinality > 0.5:
            return "ID-like (join key, not a feature)"
        if r.missing_pct > 40:
            return "mostly missing"
        return ""
    p["note"] = p.apply(note, axis=1)
    return p

t0 = time.time()
profiles = {}
for t in LOGS:
    profiles[t] = profile_log(t)
    print(f"{t}: profiled {len(profiles[t])} columns over {profiles[t].rows.iloc[0]:,} rows ({time.time() - t0:.0f}s)")
for t in LOGS:
    p = profiles[t]
    print(f"\n=== {t.upper()} log: {p.rows.iloc[0]:,} rows ===" + ("" if t != "bid" or EXACT_DISTINCT_FOR_BIDS
                                                                    else "  (distinct = approximate)"))
    display(p.drop(columns=["log", "rows", "distinct_exact"]))

bid: profiled 21 columns over 53,289,330 rows (33s)
imp: profiled 24 columns over 12,237,087 rows (44s)
clk: profiled 24 columns over 9,978 rows (44s)
conv: profiled 24 columns over 494 rows (44s)

=== BID log: 53,289,330 rows ===  (distinct = approximate)


,column,dtype,null,empty,null_text,missing,missing_pct,distinct,min,max,mean,median,cardinality,cast_failed,top_value,top_share_pct,note
0,bid_id,VARCHAR,0,0,0,0,0.00,53330251,NaN,NaN,NaN,NaN,1.0000,0,NaN,NaN,"ID-like (join key, not a feature)"
1,timestamp,TIMESTAMP,0,0,0,0,0.00,32688699,2013-06-06 00:01:04.008,2013-06-12 23:34:12.741,NaN,NaN,0.6134,0,NaN,NaN,
2,ipinyou_id,VARCHAR,0,0,23028624,23028624,43.21,15834276,NaN,NaN,NaN,NaN,0.5233,0,NaN,NaN,"ID-like (join key, not a feature)"
3,user_agent,VARCHAR,0,12399,0,12399,0.02,1096046,NaN,NaN,NaN,NaN,0.0206,12399,Mozilla/5.0 (Windows NT 5.1) AppleWebKit/537.1...,7.44,
4,ip,VARCHAR,0,2230,0,2230,0.00,733490,NaN,NaN,NaN,NaN,0.0138,2230,58.221.56.*,0.39,
5,region,BIGINT,0,0,0,0,0.00,36,-1,395,154.45,146.0,0.0000,0,216,15.37,
6,city,BIGINT,0,0,0,0,0.00,367,-1,399,159.76,158.0,0.0000,0,219,3.39,
7,ad_exchange,BIGINT,0,0,0,0,0.00,3,1,3,2.52,3.0,0.0000,0,3,63.85,
8,domain,VARCHAR,0,0,10528175,10528175,19.76,55173,NaN,NaN,NaN,NaN,0.0013,0,5F1RQS9rg5scFsf,9.31,
9,url,VARCHAR,0,0,8902368,8902368,16.71,5343605,NaN,NaN,NaN,NaN,0.1204,0,2894177023e3411e2dbb0e9b5186c2cb,1.83,



=== IMP log: 12,237,087 rows ===


,column,dtype,null,empty,null_text,missing,missing_pct,distinct,min,max,mean,median,cardinality,cast_failed,top_value,top_share_pct,note
0,bid_id,VARCHAR,0,0,0,0,0.00,12190347,NaN,NaN,NaN,NaN,0.9962,0,NaN,NaN,"ID-like (join key, not a feature)"
1,timestamp,TIMESTAMP,0,0,0,0,0.00,10155037,2013-06-06 00:01:04.192,2013-06-12 23:51:28.618,NaN,NaN,0.8299,0,NaN,NaN,
2,log_type,BIGINT,0,0,0,0,0.00,1,1,1,1.00,1.0,0.0000,0,1,100.00,constant: drop
3,ipinyou_id,VARCHAR,0,0,0,0,0.00,10146411,NaN,NaN,NaN,NaN,0.8292,0,NaN,NaN,"ID-like (join key, not a feature)"
4,user_agent,VARCHAR,0,2,1574,1576,0.01,459646,NaN,NaN,NaN,NaN,0.0376,2,Mozilla/5.0 (Windows NT 5.1) AppleWebKit/537.1...,12.24,
5,ip,VARCHAR,0,0,0,0,0.00,631255,NaN,NaN,NaN,NaN,0.0516,0,125.39.238.*,0.11,
6,region,BIGINT,0,0,0,0,0.00,35,0,395,146.18,144.0,0.0000,0,216,12.20,
7,city,BIGINT,0,0,0,0,0.00,370,0,399,151.08,145.0,0.0000,0,1,4.51,
8,ad_exchange,BIGINT,0,0,0,0,0.00,3,1,3,2.04,2.0,0.0000,0,3,35.93,
9,domain,VARCHAR,0,0,595864,595864,4.87,28504,NaN,NaN,NaN,NaN,0.0024,0,5F1RQS9rg5scFsf,9.88,



=== CLK log: 9,978 rows ===


,column,dtype,null,empty,null_text,missing,missing_pct,distinct,min,max,mean,median,cardinality,cast_failed,top_value,top_share_pct,note
0,bid_id,VARCHAR,0,0,0,0,0.00,8838,NaN,NaN,NaN,NaN,0.8857,0,NaN,NaN,"ID-like (join key, not a feature)"
1,timestamp,TIMESTAMP,0,0,0,0,0.00,9976,2013-06-06 00:01:10.312,2013-06-12 23:40:49.226,NaN,NaN,0.9998,0,NaN,NaN,
2,log_type,BIGINT,0,0,0,0,0.00,1,2,2,2.00,2.0,0.0001,0,2,100.00,constant: drop
3,ipinyou_id,VARCHAR,0,0,0,0,0.00,8912,NaN,NaN,NaN,NaN,0.8932,0,NaN,NaN,"ID-like (join key, not a feature)"
4,user_agent,VARCHAR,0,0,10,10,0.10,2444,NaN,NaN,NaN,NaN,0.2452,0,Mozilla/5.0 (Windows NT 5.1) AppleWebKit/537.1...,8.64,
5,ip,VARCHAR,0,0,0,0,0.00,8336,NaN,NaN,NaN,NaN,0.8354,0,NaN,NaN,"ID-like (join key, not a feature)"
6,region,BIGINT,0,0,0,0,0.00,35,0,395,142.75,133.0,0.0035,0,216,10.82,
7,city,BIGINT,0,0,0,0,0.00,354,0,399,147.46,138.0,0.0355,0,1,4.42,
8,ad_exchange,BIGINT,0,0,0,0,0.00,3,1,3,1.91,2.0,0.0003,0,1,42.95,
9,domain,VARCHAR,0,0,552,552,5.53,1035,NaN,NaN,NaN,NaN,0.1098,0,5F1RQS9rg5scFsf,10.99,



=== CONV log: 494 rows ===


,column,dtype,null,empty,null_text,missing,missing_pct,distinct,min,max,mean,median,cardinality,cast_failed,top_value,top_share_pct,note
0,bid_id,VARCHAR,0,0,0,0,0.00,404,NaN,NaN,NaN,NaN,0.8178,0,NaN,NaN,"ID-like (join key, not a feature)"
1,timestamp,TIMESTAMP,0,0,0,0,0.00,399,2013-06-06 00:01:32,2013-06-12 15:55:56,NaN,NaN,0.8077,0,NaN,NaN,
2,log_type,BIGINT,0,0,0,0,0.00,1,3,3,3.00,3.0,0.0020,0,3,100.00,constant: drop
3,ipinyou_id,VARCHAR,0,0,0,0,0.00,418,NaN,NaN,NaN,NaN,0.8462,0,NaN,NaN,"ID-like (join key, not a feature)"
4,user_agent,VARCHAR,0,0,6,6,1.21,196,NaN,NaN,NaN,NaN,0.4016,0,Mozilla/5.0 (Windows NT 5.1) AppleWebKit/537.1...,10.12,
5,ip,VARCHAR,0,0,0,0,0.00,402,NaN,NaN,NaN,NaN,0.8138,0,NaN,NaN,"ID-like (join key, not a feature)"
6,region,BIGINT,0,0,0,0,0.00,32,0,394,158.01,146.0,0.0648,0,216,12.15,
7,city,BIGINT,0,0,0,0,0.00,169,0,399,162.90,154.0,0.3421,0,1,3.44,
8,ad_exchange,BIGINT,0,0,0,0,0.00,3,1,3,1.54,1.0,0.0061,0,1,63.16,
9,domain,VARCHAR,0,0,31,31,6.28,153,NaN,NaN,NaN,NaN,0.3305,0,eA1XTupSMZq81YT9UDaWvpdh,11.34,


### 3. Duplicates

Two kinds of repetition:
* **Exact duplicate rows**: identical in every column. This is a logging or loading artefact, so drop them. They are counted with a 64-bit hash of the whole row, which keeps memory low.
* **Repeated bid IDs**: the same `bid_id` on rows that are otherwise different. In the click log, this is mostly one user clicking twice, which is real. In the impression log it is `bid_id` reuse (different users and times), so `bid_id` alone is not an impression key.

Bids are checked one day at a time to keep memory low.

In [6]:
# CELL 4: duplicates per log (totals and per day)
def dup_summary(t, by_day):
    allc = ", ".join(f'"{c}"' for c in COLS[t])
    key = "regexp_extract(filename, '([0-9]+)\\.txt$', 1)" if by_day else "'all days'"
    return q(f"""SELECT '{t}' AS log, {key} AS day, count(*) AS rows,
                        count(DISTINCT hash({allc})) AS distinct_rows,
                        count(DISTINCT hash(bid_id)) AS distinct_bid_ids
                 FROM raw_{t} GROUP BY ALL""")

dups = []
for t in ["imp", "clk", "conv"]:
    dups.append(dup_summary(t, False))
# bids: day by day (53M rows), then summed. A bid_id never spans two days in the bid log.
bid_days = dup_summary("bid", True)
dups.append(pd.DataFrame([{"log": "bid", "day": "all days", "rows": bid_days.rows.sum(),
                           "distinct_rows": bid_days.distinct_rows.sum(),
                           "distinct_bid_ids": bid_days.distinct_bid_ids.sum()}]))
duplicates = pd.concat(dups, ignore_index=True).set_index("log").loc[LOGS].reset_index()
duplicates["exact_duplicate_rows"] = duplicates.rows - duplicates.distinct_rows
duplicates["extra_rows_sharing_a_bid_id"] = duplicates.distinct_rows - duplicates.distinct_bid_ids
duplicates["bid_id_unique"] = duplicates.extra_rows_sharing_a_bid_id == 0
display(duplicates)

print("Per day:")
by_day = pd.concat([dup_summary(t, True) for t in ["imp", "clk", "conv"]] + [bid_days], ignore_index=True)
by_day["exact_duplicate_rows"] = by_day.rows - by_day.distinct_rows
by_day["extra_rows_sharing_a_bid_id"] = by_day.distinct_rows - by_day.distinct_bid_ids
display(by_day.pivot_table(index="day", columns="log",
                           values=["exact_duplicate_rows", "extra_rows_sharing_a_bid_id"], fill_value=0).astype(int))

# What differs when an impression bid_id repeats? (only the repeated IDs are examined)
display(q("""
    WITH rep AS (SELECT bid_id FROM typed_imp GROUP BY 1 HAVING count(*) > 1),
    r AS (SELECT bid_id, count(*) AS n, count(DISTINCT timestamp) AS n_ts, count(DISTINCT ipinyou_id) AS n_user,
                 count(DISTINCT advertiser_id) AS n_adv, count(DISTINCT paying_price) AS n_pay,
                 count(DISTINCT log_date) AS n_days
          FROM typed_imp JOIN rep USING (bid_id) GROUP BY 1)
    SELECT count(*) AS repeated_impression_bid_ids, sum(n) AS rows_involved,
           sum((n_ts > 1)::INT) AS differ_in_timestamp, sum((n_user > 1)::INT) AS differ_in_user,
           sum((n_adv > 1)::INT) AS differ_in_advertiser, sum((n_pay > 1)::INT) AS differ_in_paying_price,
           sum((n_days > 1)::INT) AS span_several_days
    FROM r""").T.rename(columns={0: "value"}))

,log,day,rows,distinct_rows,distinct_bid_ids,exact_duplicate_rows,extra_rows_sharing_a_bid_id,bid_id_unique
0,bid,all days,53289330,53289330,53289330,0,0,True
1,imp,all days,12237087,12237081,12190347,6,46734,False
2,clk,all days,9978,9978,8838,0,1140,False
3,conv,all days,494,418,404,76,14,False


Per day:


exact_duplicate_rows              extra_rows_sharing_a_bid_id                
log                       bid clk conv imp                         bid  clk conv   imp
day                                                                                   
20130606                    0   0    5   0                           0  130    0  6275
20130607                    0   0    7   2                           0  110    0  5844
20130608                    0   0   17   4                           0  139    0  5518
20130609                    0   0    2   0                           0  107    1  5968
20130610                    0   0   26   0                           0  221    3  8985
20130611                    0   0    9   0                           0  195    2  6346
20130612                    0   0   10   0                           0  238    8  7794

,value
repeated_impression_bid_ids,45044.0
rows_involved,91784.0
differ_in_timestamp,44983.0
differ_in_user,39358.0
differ_in_advertiser,0.0
differ_in_paying_price,161.0
span_several_days,4.0


### 4. Value ranges and impossible values

Each check below should be zero, or have a clear explanation:
* **paying price > bidding price**: impossible in a pure second-price auction
* **paying price < floor price**: the floor should be the minimum price
* **region/city = 0 or −1**: "unknown" codes
* **timestamp outside the file's day**, or **unparseable**
* **negative prices**, and **zero paying price**

The value lists for the coded columns (ad exchange, slot visibility and format, slot sizes, bid price levels) show what each code is and how often it appears.

In [7]:
# CELL 5: impossible / suspicious values, all logs
checks = []
for t in LOGS:
    has_pay = "paying_price" in COLS[t]
    r = q(f"""SELECT count(*) AS rows,
                count(*) FILTER (WHERE timestamp IS NULL) AS bad_timestamp,
                count(*) FILTER (WHERE strftime(timestamp, '%Y%m%d') <> log_date) AS timestamp_not_on_file_day,
                count(*) FILTER (WHERE bidding_price < 0 OR ad_slot_floor_price < 0) AS negative_price,
                count(*) FILTER (WHERE region <= 0) AS region_unknown,
                count(*) FILTER (WHERE city <= 0) AS city_unknown,
                count(*) FILTER (WHERE ad_slot_width <= 0 OR ad_slot_height <= 0) AS bad_slot_size,
                count(*) FILTER (WHERE bidding_price < ad_slot_floor_price) AS bid_below_floor
                {", count(*) FILTER (WHERE paying_price > bidding_price) AS paying_above_bid" if has_pay else ""}
                {", count(*) FILTER (WHERE paying_price < ad_slot_floor_price) AS paying_below_floor" if has_pay else ""}
                {", count(*) FILTER (WHERE paying_price = 0) AS paying_zero" if has_pay else ""}
              FROM typed_{t}""").iloc[0].to_dict()
    r["log"] = t
    checks.append(r)
value_checks = pd.DataFrame(checks).set_index("log").fillna(0).astype("int64")
display(value_checks.T)

print("How far above the bid does the paying price go (impressions where paying > bidding)?")
display(q("""SELECT advertiser_id, count(*) AS impressions,
                    median(paying_price - bidding_price) AS median_excess, max(paying_price - bidding_price) AS max_excess,
                    round(100.0 * count(*) / (SELECT count(*) FROM typed_imp i2 WHERE i2.advertiser_id = i.advertiser_id), 2) AS pct_of_campaign
             FROM typed_imp i WHERE paying_price > bidding_price GROUP BY 1 ORDER BY 1"""))

for c in ["ad_exchange", "ad_slot_visibility", "ad_slot_format", "bidding_price"]:
    print(f"\n{c}: values in the impression log")
    display(q(f"""SELECT "{c}", count(*) AS impressions, round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
                  FROM typed_imp GROUP BY 1 ORDER BY 1"""))
print("\nSlot sizes (width x height), top 15:")
display(q("""SELECT ad_slot_width || 'x' || ad_slot_height AS slot_size, count(*) AS impressions,
                    round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
             FROM typed_imp GROUP BY 1 ORDER BY 2 DESC LIMIT 15"""))
print("\nUser tags: comma-separated lists. Distinct tags and how many impressions carry any tag:")
display(q("""SELECT (SELECT count(DISTINCT tag) FROM (SELECT unnest(string_split(user_tags, ',')) AS tag
                                                    FROM typed_imp WHERE user_tags IS NOT NULL)) AS distinct_tags,
                    round(100.0 * count(user_tags) / count(*), 2) AS pct_impressions_with_tags FROM typed_imp"""))

log,bid,imp,clk,conv
rows,53289330,12237087,9978,494
bad_timestamp,0,0,0,0
timestamp_not_on_file_day,0,0,0,0
negative_price,0,0,0,0
region_unknown,682685,210680,278,12
city_unknown,682685,210680,278,12
bad_slot_size,0,0,0,0
bid_below_floor,0,0,0,0
paying_above_bid,0,191091,0,0
paying_below_floor,0,64113,5692,182


How far above the bid does the paying price go (impressions where paying > bidding)?


,advertiser_id,impressions,median_excess,max_excess,pct_of_campaign
0,3358,84412,12.0,40,4.85
1,3427,81326,14.0,40,3.14
2,3476,25353,12.0,29,1.29



ad_exchange: values in the impression log


,ad_exchange,impressions,pct
0,1,3878242,31.69
1,2,3962486,32.38
2,3,4396359,35.93



ad_slot_visibility: values in the impression log


... (5 outputs left out)


Slot sizes (width x height), top 15:


,slot_size,impressions,pct
0,1000x90,3117964,25.48
1,300x250,3098816,25.32
2,728x90,1965880,16.06
3,336x280,1297599,10.60
4,160x600,686394,5.61
5,250x250,574233,4.69
6,950x90,558901,4.57
7,960x90,291763,2.38
8,200x200,289714,2.37
9,468x60,276551,2.26



User tags: comma-separated lists. Distinct tags and how many impressions carry any tag:


,distinct_tags,pct_impressions_with_tags
0,43,86.18


### 5. The data profile table

One row per log and column, combining everything above: **this is the step 1 output**. Saved as `data/processed/data_profile.csv` (a few KB). The log-level facts (rows, days, duplicates) are repeated on every row, so the table stands on its own.

In [8]:
# CELL 6: the data profile table
data_profile = pd.concat(profiles.values(), ignore_index=True)
log_facts = duplicates.set_index("log")[["exact_duplicate_rows", "extra_rows_sharing_a_bid_id"]]
log_facts["days"] = [int((rows_per_day.drop(index="TOTAL")[t] > 0).sum()) for t in log_facts.index]
data_profile = data_profile.merge(log_facts, left_on="log", right_index=True, how="left")
data_profile = data_profile[["log", "column", "dtype", "rows", "days", "missing", "missing_pct", "null", "empty",
                             "null_text", "cast_failed", "distinct", "distinct_exact", "cardinality", "min", "max", "mean", "median",
                             "top_value", "top_share_pct", "exact_duplicate_rows", "extra_rows_sharing_a_bid_id", "note"]]
display(data_profile)

try:
    out = PROJECT_ROOT / "data" / "processed" / "data_profile.csv"
    data_profile.to_csv(out, index=False)
    print("Saved", out.relative_to(PROJECT_ROOT))
except OSError as e:
    print("Could not save the CSV (disk full?):", e)

# Columns to keep for reporting (the impression log is the analysis table)
imp_p = data_profile[data_profile.log == "imp"]
drop = imp_p[imp_p.note.str.contains("drop")]["column"].tolist()
ids = imp_p[imp_p.note.str.startswith("ID-like")]["column"].tolist()
print("Drop (no information):", drop)
print("Keep only as join keys (ID-like):", ids)

,log,column,dtype,rows,days,missing,missing_pct,null,empty,null_text,cast_failed,distinct,distinct_exact,cardinality,min,max,mean,median,top_value,top_share_pct,exact_duplicate_rows,extra_rows_sharing_a_bid_id,note
0,bid,bid_id,VARCHAR,53289330,7,0,0.00,0,0,0,0,53330251,False,1.0000,NaN,NaN,NaN,NaN,NaN,NaN,0,0,"ID-like (join key, not a feature)"
1,bid,timestamp,TIMESTAMP,53289330,7,0,0.00,0,0,0,0,32688699,False,0.6134,2013-06-06 00:01:04.008,2013-06-12 23:34:12.741,NaN,NaN,NaN,NaN,0,0,
2,bid,ipinyou_id,VARCHAR,53289330,7,23028624,43.21,0,0,23028624,0,15834276,False,0.5233,NaN,NaN,NaN,NaN,NaN,NaN,0,0,"ID-like (join key, not a feature)"
3,bid,user_agent,VARCHAR,53289330,7,12399,0.02,0,12399,0,12399,1096046,False,0.0206,NaN,NaN,NaN,NaN,Mozilla/5.0 (Windows NT 5.1) AppleWebKit/537.1...,7.44,0,0,
4,bid,ip,VARCHAR,53289330,7,2230,0.00,0,2230,0,2230,733490,False,0.0138,NaN,NaN,NaN,NaN,58.221.56.*,0.39,0,0,
5,bid,region,BIGINT,53289330,7,0,0.00,0,0,0,0,36,False,0.0000,-1,395,154.45,146.0,216,15.37,0,0,
6,bid,city,BIGINT,53289330,7,0,0.00,0,0,0,0,367,False,0.0000,-1,399,159.76,158.0,219,3.39,0,0,
7,bid,ad_exchange,BIGINT,53289330,7,0,0.00,0,0,0,0,3,False,0.0000,1,3,2.52,3.0,3,63.85,0,0,
8,bid,domain,VARCHAR,53289330,7,10528175,19.76,0,0,10528175,0,55173,False,0.0013,NaN,NaN,NaN,NaN,5F1RQS9rg5scFsf,9.31,0,0,
9,bid,url,VARCHAR,53289330,7,8902368,16.71,0,0,8902368,0,5343605,False,0.1204,NaN,NaN,NaN,NaN,2894177023e3411e2dbb0e9b5186c2cb,1.83,0,0,


Saved data/processed/data_profile.csv
Drop (no information): ['log_type', 'anonymous_url_id']
Keep only as join keys (ID-like): ['bid_id', 'ipinyou_id']


## Findings

Generated from the numbers above. 

In [10]:
# CELL 7: findings
tot = rows_per_day.loc["TOTAL"]
d = duplicates.set_index("log")
vc = value_checks
bp = data_profile[(data_profile.log == "bid")].set_index("column")
F = [
    f"**Coverage.** {len(days_found)} days ({days_found[0]}–{days_found[-1]}) in every log: {tot['bid']:,} bids, "
    f"{tot['imp']:,} impressions, {tot['clk']:,} clicks and {tot['conv']:,} conversions across "
    f"{len(rows_per_adv) - 1} campaigns. Confidence: high (exact counts).",
    "**Columns with no information:** " + (", ".join(f"`{c}`" for c in drop) or "none") +
    f". **ID-like columns** (join keys, not features): " + ", ".join(f"`{c}`" for c in ids) + ".",
    f"**Missing values.** In impressions, the most-missing columns are " + ", ".join(
        f"`{r.column}` ({r.missing_pct:.1f}%)" for r in imp_p.sort_values("missing_pct", ascending=False).head(3).itertuples()) +
    f". In bids, `ipinyou_id` is missing on {bp.loc['ipinyou_id', 'missing_pct']:.1f}% of rows and `user_tags` on "
    f"{bp.loc['user_tags', 'missing_pct']:.0f}%. Of all missing values, "
    f"{data_profile.null_text.sum() / max(data_profile.missing.sum(), 1):.0%} are the literal text `null`, "
    f"{data_profile["empty"].sum() / max(data_profile.missing.sum(), 1):.0%} empty fields"
    + (f", and {int(data_profile.cast_failed.sum()):,} values failed type conversion: check those." if data_profile.cast_failed.sum()
       else ", and no value was lost in type conversion."),
    f"**Duplicates.** Exact duplicate rows: {d.loc['imp', 'exact_duplicate_rows']:,} impressions, "
    f"{d.loc['clk', 'exact_duplicate_rows']:,} clicks, {d.loc['conv', 'exact_duplicate_rows']:,} conversions, "
    f"{d.loc['bid', 'exact_duplicate_rows']:,} bids: drop them. Repeated bid IDs on otherwise different rows: "
    f"{d.loc['imp', 'extra_rows_sharing_a_bid_id']:,} impressions (bid_id reuse, so the impression key needs user + "
    f"timestamp too), {d.loc['clk', 'extra_rows_sharing_a_bid_id']:,} clicks (repeat clicks), "
    f"{d.loc['bid', 'extra_rows_sharing_a_bid_id']:,} bids.",
    f"**Impossible values.** {vc.loc['imp', 'paying_above_bid']:,} impressions "
    f"({vc.loc['imp', 'paying_above_bid'] / vc.loc['imp', 'rows']:.1%}) paid more than the bid and "
    f"{vc.loc['imp', 'paying_below_floor']:,} paid below the floor. Kept, because cost must match the dataset paper, "
    f"but flagged. {vc.loc['imp', 'region_unknown']:,} impressions have an unknown region (0). Timestamps: "
    f"{int(vc.bad_timestamp.sum()):,} unparseable, {int(vc.timestamp_not_on_file_day.sum()):,} outside their file's day.",
]
display(Markdown("\n".join(f"{i}. {t}" for i, t in enumerate(F, 1))))

1. **Coverage.** 7 days (20130606–20130612) in every log: 53,289,330 bids, 12,237,087 impressions, 9,978 clicks and 494 conversions across 5 campaigns. Confidence: high (exact counts).
2. **Columns with no information:** `log_type`, `anonymous_url_id`. **ID-like columns** (join keys, not features): `bid_id`, `ipinyou_id`.
3. **Missing values.** In impressions, the most-missing columns are `anonymous_url_id` (100.0%), `user_tags` (13.8%), `domain` (4.9%). In bids, `ipinyou_id` is missing on 43.2% of rows and `user_tags` on 100%. Of all missing values, 89% are the literal text `null`, 11% empty fields, and 17,694,285 values failed type conversion: check those.
4. **Duplicates.** Exact duplicate rows: 6 impressions, 0 clicks, 76 conversions, 0 bids: drop them. Repeated bid IDs on otherwise different rows: 46,734 impressions (bid_id reuse, so the impression key needs user + timestamp too), 1,140 clicks (repeat clicks), 0 bids.
5. **Impossible values.** 191,091 impressions (1.6%) paid more than the bid and 64,113 paid below the floor. Kept, because cost must match the dataset paper, but flagged. 210,680 impressions have an unknown region (0). Timestamps: 0 unparseable, 0 outside their file's day.